# Colab P1 Extensions Notebook

Notebook one-shot cho 4 nhom phep do nang cap muc P1 trong plan hoan thien:

1. **P1.1 -- Validate mAP cho ONNX + TensorRT**: lay model `.onnx` va `.engine` da export, chay `model.val()` voi cung protocol Muc 4.2 cua bao cao de do truot mAP do luong tu (FP16).
2. **P1.2 -- Multi-seed training**: lap lai student baseline va student KD voi 3 seed khac nhau (42, 123, 456) tren cung dataset/epochs/imgsz, tinh mean +- std, va tinh paired-difference (KD - baseline) co khoang tin cay.
3. **P1.3 -- Per-class mAP**: 5 lop (bicycle/bus/car/motorbike/person) cho teacher, student baseline (mean over seeds) va student KD (mean over seeds).
4. **P1.4 -- Anh dinh tinh**: 5 anh model du doan tot (Input | Ground Truth | Prediction) lay tu val set.

Chay tu tren xuong duoi 1 lan. Cuoi notebook tu dong zip va tai ve `p1_results.zip`.

Chi phi du kien tren T4: ~1.5-2.5h GPU (~4-6 CU = $0.40-0.60). Neu da co teacher pre-trained tu bundle (URL hoac upload), chi mat ~1h.


## 1. CONFIG

In [ ]:
# =====================
# 1. CONFIG
# =====================
GITHUB_REPO_URL = "https://github.com/hoaianthai345/HPC_Nhom1_MLOps_ObjectDectection.git"
PROJECT_DIR = "/content/HPC_Nhom1_MLOps_ObjectDectection"

ULTRALYTICS_KD_REPO_URL = "https://github.com/ultralytics/ultralytics.git"
ULTRALYTICS_KD_DIR = "/content/ultralytics-kd"

KAGGLE_DATASET = "yusufberksardoan/traffic-detection-project"
RAW_DATA_DIR = "/content/data/raw"
SUBSET_DATA_DIR = "/content/data/demo_subset"
USE_SUBSET = True
SUBSET_SIZE = 1000

TEACHER_MODEL = "yolo26x.pt"
STUDENT_MODEL = "yolo26n.pt"
IMG_SIZE = 640

# === P1.2 MULTI-SEED ===
SEEDS = [42, 123, 456]
MULTISEED_EPOCHS = 30
MULTISEED_BATCH  = 16

# === P1.1 VALIDATE FORMAT ===
VAL_CONF = 0.001
VAL_IOU  = 0.7

# === P1.4 QUALITATIVE ===
N_QUALITATIVE = 5

# === Optional: bundle pre-train (tiet kiem 72 phut teacher) ===
# Co the:
#   - dat URL .zip cua benchmark_bundle (chua teacher_best.pt, serving_model.{onnx,engine}, ...)
#   - hoac de trong de notebook hoi upload thu cong qua files.upload()
#   - hoac neu trong cung session ban da chay P2 va co /content/model_artifacts -> tu dong dung
PRETRAINED_BUNDLE_URL = ""

ARTIFACT_DIR = "/content/model_artifacts"
RESULTS_DIR  = "/content/p1_results"
print("Config loaded")

## 2. Runtime + device fingerprint

In [ ]:
# =====================
# 2. RUNTIME + DEVICE
# =====================
import os, sys, json, time, shlex, shutil, platform, subprocess, statistics
from pathlib import Path

PYTHON = shlex.quote(sys.executable)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(ARTIFACT_DIR, exist_ok=True)

def run(cmd, cwd=None, check=True):
    print(f"$ {cmd}")
    return subprocess.run(cmd, shell=True, cwd=cwd, check=check)

def pip_cmd(args, check=True):
    ip = globals().get("get_ipython", lambda: None)()
    if ip is not None:
        try: return ip.run_line_magic("pip", args)
        except Exception:
            if check: raise
            return None
    return run(f"{PYTHON} -m pip {args}", check=check)

DEVICE_INFO = {}
try:
    import torch
    DEVICE = "0" if torch.cuda.is_available() else "cpu"
    DEVICE_INFO["cuda"] = torch.cuda.is_available()
    DEVICE_INFO["torch"] = torch.__version__
    if torch.cuda.is_available():
        DEVICE_INFO["gpu"] = torch.cuda.get_device_name(0)
        DEVICE_INFO["gpu_mem_gb"] = round(torch.cuda.get_device_properties(0).total_memory/1e9, 2)
except Exception as e:
    DEVICE = "cpu"; DEVICE_INFO["torch_err"] = str(e)
DEVICE_INFO["python"] = platform.python_version()
DEVICE_INFO["platform"] = platform.platform()
DEVICE_INFO["cpu_count"] = os.cpu_count()
try:
    with open("/proc/meminfo") as f:
        for line in f:
            if line.startswith("MemTotal"):
                DEVICE_INFO["ram_gb"] = round(int(line.split()[1])/1e6, 2); break
except Exception: pass

print("DEVICE:", DEVICE)
print(json.dumps(DEVICE_INFO, indent=2, ensure_ascii=False))

## 3. Clone repo + Ultralytics KD trainer + deps

In [ ]:
# =====================
# 3. CLONE + DEPS + KD TRAINER
# =====================
if Path(PROJECT_DIR).exists():
    run("git pull", cwd=PROJECT_DIR, check=False)
else:
    run(f"git clone {GITHUB_REPO_URL} {PROJECT_DIR}")
os.chdir(PROJECT_DIR)

pip_cmd("install --upgrade pip")
pip_cmd("install kaggle pyyaml mlflow boto3 botocore pillow opencv-python tqdm onnx onnxruntime-gpu matplotlib scipy")
for req in ["data_pipeline/requirements.txt", "serving_pipeline/requirements.txt"]:
    if Path(req).exists():
        pip_cmd(f"install -r {req}", check=False)

if Path(ULTRALYTICS_KD_DIR).exists():
    run("git pull", cwd=ULTRALYTICS_KD_DIR, check=False)
else:
    run(f"git clone {ULTRALYTICS_KD_REPO_URL} {ULTRALYTICS_KD_DIR}")

custom_trainer = next((p for p in [Path(PROJECT_DIR)/"notebooks"/"trainer.py", Path(PROJECT_DIR)/"trainer.py"] if p.exists()), None)
assert custom_trainer is not None, "Khong tim thay trainer.py custom KD trong repo"
target_trainer = Path(ULTRALYTICS_KD_DIR)/"ultralytics"/"engine"/"trainer.py"
shutil.copy2(custom_trainer, target_trainer)
print("Copied KD trainer ->", target_trainer)

pip_cmd(f"install -e {ULTRALYTICS_KD_DIR}")
ultra_src = str(Path(ULTRALYTICS_KD_DIR).resolve())
if ultra_src not in sys.path: sys.path.insert(0, ultra_src)
import importlib; importlib.invalidate_caches()
import ultralytics
from ultralytics import YOLO
print("Ultralytics:", ultralytics.__file__)

## 4. Kaggle + dataset

In [ ]:
# =====================
# 4. DATASET
# =====================
import yaml
kaggle_json = Path.home()/".kaggle"/"kaggle.json"
if not kaggle_json.exists():
    from google.colab import files
    print("Upload kaggle.json"); up = files.upload()
    assert "kaggle.json" in up, "Can upload kaggle.json"
    kaggle_json.parent.mkdir(parents=True, exist_ok=True)
    shutil.move("kaggle.json", kaggle_json); kaggle_json.chmod(0o600)

Path(RAW_DATA_DIR).mkdir(parents=True, exist_ok=True)
r = subprocess.run(f"{PYTHON} -m data_pipeline kaggle download --dataset {KAGGLE_DATASET} --output {RAW_DATA_DIR} --organize", shell=True, cwd=PROJECT_DIR)
if r.returncode != 0:
    run(f"kaggle datasets download -d {KAGGLE_DATASET} -p {RAW_DATA_DIR} --unzip")

DATA_ROOT = RAW_DATA_DIR
if USE_SUBSET:
    rs = subprocess.run(f"{PYTHON} -m data_pipeline dataset subset --input {RAW_DATA_DIR} --output {SUBSET_DATA_DIR} --size {SUBSET_SIZE}", shell=True, cwd=PROJECT_DIR)
    if rs.returncode == 0: DATA_ROOT = SUBSET_DATA_DIR

def find_data_yaml(root):
    root = Path(root)
    c = list(root.rglob("data.yaml")) + list(root.rglob("dataset.yaml"))
    return sorted(c, key=lambda p: len(p.parts))[0] if c else None

data_yaml = find_data_yaml(DATA_ROOT) or find_data_yaml(RAW_DATA_DIR)
assert data_yaml is not None
print("DATA_YAML:", data_yaml)

## 5. Tai bundle pre-train (de bo qua train teacher)\n\nUu tien theo thu tu: (1) `PRETRAINED_BUNDLE_URL`, (2) artifact da co trong `/content/model_artifacts/` (P2 truoc do), (3) upload thu cong, (4) train lai tu dau.

In [ ]:
# =====================
# 5. PRELOAD BUNDLE (optional)
# =====================
def newest(paths):
    paths = [p for p in paths if p.exists()]
    return sorted({p.resolve() for p in paths}, key=lambda p: p.stat().st_mtime, reverse=True)[0] if paths else None

teacher_best = Path(ARTIFACT_DIR)/"teacher_best.pt"
onnx_path    = Path(ARTIFACT_DIR)/"serving_model.onnx"
engine_path  = Path(ARTIFACT_DIR)/"serving_model.engine"

if PRETRAINED_BUNDLE_URL:
    print("[5] Tai bundle tu URL...")
    run(f"wget -q -O /content/bundle.zip {PRETRAINED_BUNDLE_URL}")
    run("unzip -o /content/bundle.zip -d /content/")

# Ho tro ca cau truc /content/model_artifacts/ va /content/benchmark_bundle/model_artifacts/
for src in [Path("/content/benchmark_bundle/model_artifacts"), Path("/content/model_artifacts")]:
    if src.exists():
        for f in src.glob("*"):
            dst = Path(ARTIFACT_DIR)/f.name
            if not dst.exists():
                shutil.copy2(f, dst)

# Neu van chua co teacher -> hoi upload
if not teacher_best.exists():
    print("[5] Khong co teacher pre-train. Co the upload teacher_best.pt + (serving_model.onnx, serving_model.engine) tuy chon...")
    try:
        from google.colab import files
        up = files.upload()
        for fname, content in up.items():
            (Path(ARTIFACT_DIR)/fname).write_bytes(content)
    except Exception as e:
        print("Upload bo qua:", e)

print("\n[5] Artifact san sang:")
for f in sorted(Path(ARTIFACT_DIR).glob("*")):
    print(f"  - {f.name}: {f.stat().st_size/1048576:.2f} MB")

HAVE_TEACHER = teacher_best.exists()
HAVE_ONNX    = onnx_path.exists()
HAVE_ENGINE  = engine_path.exists()

if not HAVE_TEACHER:
    print("\n[5] Khong co teacher -> se train tu dau (toi them ~72 phut)")
else:
    print("\n[5] OK -- bo qua train teacher")

## 6. Train teacher (chi chay khi chua co bundle)

In [ ]:
# =====================
# 6. TRAIN TEACHER (skip neu da co)
# =====================
if not HAVE_TEACHER:
    teacher_cmd = (f"{PYTHON} scripts/train_teacher_model.py --data {data_yaml} --model {TEACHER_MODEL} "
                   f"--epochs 30 --batch 4 --imgsz {IMG_SIZE} --device {DEVICE} "
                   f"--project runs/teacher --name teacher_yolo --no-mlflow")
    run(teacher_cmd, cwd=PROJECT_DIR)
    found = newest([*Path(PROJECT_DIR).glob('runs/**/teacher_yolo*/weights/best.pt'),
                    *Path(ULTRALYTICS_KD_DIR).glob('runs/**/teacher_yolo*/weights/best.pt')])
    assert found, "Khong tim thay teacher best.pt sau khi train"
    shutil.copy2(found, teacher_best)
    HAVE_TEACHER = True
print("teacher_best:", teacher_best)

## P1.2 -- Multi-seed training (3 seed x [student baseline + student KD])\n\nMoi seed chay lan luot baseline (in-process) va KD (subprocess). Tong: 6 run x ~12 phut = ~72 phut tren T4.

In [ ]:
# =====================
# P1.2 MULTI-SEED TRAINING
# =====================
import torch

base_kd_cfg = Path(PROJECT_DIR)/"training_pipeline/src/config/train.yaml"
seed_dir = Path("/content/seed_runs"); seed_dir.mkdir(exist_ok=True)

def read_train_metrics(pt):
    ck = torch.load(pt, map_location="cpu", weights_only=False)
    tm = ck.get("train_metrics", {}) or {}
    tt = ck.get("train_results", {}).get("time", [None])
    return {
        "precision":  tm.get("metrics/precision(B)"),
        "recall":     tm.get("metrics/recall(B)"),
        "map50":      tm.get("metrics/mAP50(B)"),
        "map5095":    tm.get("metrics/mAP50-95(B)"),
        "train_time_s": tt[-1] if tt else None,
    }

SEED_RESULTS = {"baseline": {}, "kd": {}}

for seed in SEEDS:
    print(f"\n========== SEED {seed} ==========")

    # --- Student baseline (in-process de output bat duoc) ---
    print(f"[seed {seed}] STUDENT BASELINE")
    student = YOLO(STUDENT_MODEL)
    sr = student.train(data=str(data_yaml), epochs=MULTISEED_EPOCHS, batch=MULTISEED_BATCH,
                       imgsz=IMG_SIZE, device=DEVICE, seed=seed,
                       project=f"runs/seed_baseline_{seed}", name=f"baseline_s{seed}",
                       save=True, plots=False, verbose=False)
    bl_best = Path(sr.save_dir)/"weights"/"best.pt"
    SEED_RESULTS["baseline"][seed] = {"seed": seed, "checkpoint": str(bl_best), **read_train_metrics(bl_best)}
    print(f"  baseline s{seed}: {SEED_RESULTS['baseline'][seed]}")

    # --- Student KD (subprocess voi yaml rieng cho seed) ---
    print(f"[seed {seed}] STUDENT KD")
    cfg = yaml.safe_load(base_kd_cfg.read_text())
    cfg.setdefault("training", {}).update({"epochs": MULTISEED_EPOCHS, "batch": MULTISEED_BATCH,
                                            "imgsz": IMG_SIZE, "device": DEVICE, "seed": seed})
    cfg.setdefault("logging", {}).update({"project": f"runs/seed_kd_{seed}", "name": f"kd_s{seed}"})
    cfg_yaml = seed_dir/f"train_kd_s{seed}.yaml"
    cfg_yaml.write_text(yaml.safe_dump(cfg, sort_keys=False))

    cmd = (f"{PYTHON} training_pipeline/src/train.py {cfg_yaml} "
           f"--teacher-weights {teacher_best} --student-weights {STUDENT_MODEL} "
           f"--data {data_yaml} --mlflow-tracking-uri file:/content/mlruns "
           f"--mlflow-experiment multiseed --mlflow-run-name kd_s{seed}")
    run(cmd, cwd=PROJECT_DIR, check=False)
    kd_cand = newest([*Path(PROJECT_DIR).glob(f"runs/**/kd_s{seed}*/weights/best.pt"),
                      *Path(ULTRALYTICS_KD_DIR).glob(f"runs/**/kd_s{seed}*/weights/best.pt")])
    if kd_cand is None:
        print(f"  [WARN] khong tim thay kd s{seed} best.pt"); continue
    SEED_RESULTS["kd"][seed] = {"seed": seed, "checkpoint": str(kd_cand), **read_train_metrics(kd_cand)}
    print(f"  kd s{seed}: {SEED_RESULTS['kd'][seed]}")

# Aggregate
import numpy as np
def agg(rows, metric):
    xs = [r[metric] for r in rows.values() if r.get(metric) is not None]
    return (round(float(np.mean(xs)), 4), round(float(np.std(xs, ddof=1)), 4)) if len(xs) > 1 else (xs[0] if xs else None, 0.0)

AGG = {}
for grp in ["baseline", "kd"]:
    AGG[grp] = {}
    for m in ["map50","map5095","precision","recall","train_time_s"]:
        mu, sd = agg(SEED_RESULTS[grp], m)
        AGG[grp][m+"_mean"] = mu
        AGG[grp][m+"_std"]  = sd

# Paired difference KD - baseline (over same seeds)
DIFFS = {}
for m in ["map50","map5095","precision","recall"]:
    pairs = []
    for s in SEEDS:
        if s in SEED_RESULTS["kd"] and s in SEED_RESULTS["baseline"]:
            a = SEED_RESULTS["kd"][s].get(m); b = SEED_RESULTS["baseline"][s].get(m)
            if a is not None and b is not None: pairs.append(a - b)
    if pairs:
        DIFFS[m] = {"deltas": [round(x, 4) for x in pairs],
                     "mean": round(float(np.mean(pairs)), 4),
                     "std":  round(float(np.std(pairs, ddof=1)) if len(pairs)>1 else 0.0, 4),
                     "n_seeds": len(pairs)}

# Save
(Path(RESULTS_DIR)/"seed_results.json").write_text(json.dumps({
    "seeds": SEEDS, "epochs": MULTISEED_EPOCHS,
    "per_seed": SEED_RESULTS, "aggregate": AGG, "kd_minus_baseline": DIFFS,
}, indent=2, ensure_ascii=False))

# Print Markdown
print("\n=== Bang multi-seed (mean +- std qua", len(SEEDS), "seed) ===")
print("| Model | Precision | Recall | mAP50 | mAP50-95 | Train (s) |")
print("|---|---:|---:|---:|---:|---:|")
for grp_label, grp in [("Student baseline","baseline"),("Student KD","kd")]:
    a = AGG[grp]
    print(f"| {grp_label} | {a['precision_mean']:.3f} $\\pm$ {a['precision_std']:.3f} | "
          f"{a['recall_mean']:.3f} $\\pm$ {a['recall_std']:.3f} | "
          f"{a['map50_mean']:.3f} $\\pm$ {a['map50_std']:.3f} | "
          f"{a['map5095_mean']:.3f} $\\pm$ {a['map5095_std']:.3f} | "
          f"{a['train_time_s_mean']:.0f} |")

print("\n=== Paired difference KD - baseline (cung seed) ===")
for m, v in DIFFS.items():
    print(f"  {m}: mean Delta = {v['mean']:+.4f} (std {v['std']:.4f}, n={v['n_seeds']})")

## 7. Export ONNX + TensorRT tu mot checkpoint KD (cho P1.1)\n\nDung checkpoint cua seed dau tien (hoac bundle co san) lam serving model de val mAP cho 3 dinh dang.

In [ ]:
# =====================
# 7. EXPORT ONNX + TENSORRT
# =====================
# Chon checkpoint KD goc cho serving (tu seed dau tien neu vua train)
if SEEDS[0] in SEED_RESULTS["kd"]:
    serving_pt = Path(ARTIFACT_DIR)/"serving_model.pt"
    shutil.copy2(SEED_RESULTS["kd"][SEEDS[0]]["checkpoint"], serving_pt)
    print(f"[7] serving_model.pt copied tu seed {SEEDS[0]}")
else:
    serving_pt = teacher_best
    print(f"[7] WARN: dung teacher_best.pt lam serving model (khong co KD checkpoint)")

# Re-export neu chua co
if not HAVE_ONNX or not (Path(ARTIFACT_DIR)/"serving_model.onnx").exists():
    try:
        YOLO(str(serving_pt)).export(format="onnx", imgsz=IMG_SIZE, dynamic=False, simplify=True)
        onnx_path = Path(ARTIFACT_DIR)/"serving_model.onnx"
        # Move/copy from default location if needed
        for cand in [serving_pt.parent/"serving_model.onnx", Path("/content/model_artifacts/serving_model.onnx")]:
            if cand.exists() and cand.resolve() != onnx_path.resolve():
                shutil.copy2(cand, onnx_path); break
        HAVE_ONNX = onnx_path.exists()
        print(f"[7] ONNX exported -> {onnx_path}")
    except Exception as e:
        print(f"[7] ONNX export failed: {e}")

if not HAVE_ENGINE or not (Path(ARTIFACT_DIR)/"serving_model.engine").exists():
    try:
        pip_cmd("install tensorrt", check=False)
        YOLO(str(serving_pt)).export(format="engine", imgsz=IMG_SIZE, half=True)
        engine_path = Path(ARTIFACT_DIR)/"serving_model.engine"
        for cand in [serving_pt.parent/"serving_model.engine", Path("/content/model_artifacts/serving_model.engine")]:
            if cand.exists() and cand.resolve() != engine_path.resolve():
                shutil.copy2(cand, engine_path); break
        HAVE_ENGINE = engine_path.exists()
        print(f"[7] TensorRT engine exported -> {engine_path}")
    except Exception as e:
        print(f"[7] TensorRT export failed (bo qua): {e}")

print(f"\n[7] Status: PT={serving_pt.exists()} ONNX={HAVE_ONNX} ENGINE={HAVE_ENGINE}")

## P1.1 + P1.3 -- Validate mAP + per-class cho 3 dinh dang trien khai\n\nChay `model.val()` chuan Ultralytics cho `.pt`, `.onnx` va `.engine`. Cung cau hinh `conf=0.001, iou=0.7, imgsz=640`. Trich them per-class mAP.

In [ ]:
# =====================
# P1.1 + P1.3 VALIDATE FORMATS
# =====================
CLASS_NAMES = ["bicycle","bus","car","motorbike","person"]

FORMAT_VAL = {}
targets = [("pytorch_pt", serving_pt)]
if HAVE_ONNX:   targets.append(("onnx",     Path(ARTIFACT_DIR)/"serving_model.onnx"))
if HAVE_ENGINE: targets.append(("trt_fp16", Path(ARTIFACT_DIR)/"serving_model.engine"))

for label, mp in targets:
    print(f"\n[val] {label}: {mp.name} ({mp.stat().st_size/1048576:.2f} MB)")
    try:
        m = YOLO(str(mp), task="detect")
        r = m.val(data=str(data_yaml), imgsz=IMG_SIZE, conf=VAL_CONF, iou=VAL_IOU,
                  split="val", device=DEVICE, verbose=False)
        entry = {
            "format": label,
            "size_mb": round(mp.stat().st_size/1048576, 2),
            "map50":   round(float(r.box.map50), 4),
            "map5095": round(float(r.box.map), 4),
            "precision": round(float(r.box.mp), 4),
            "recall":  round(float(r.box.mr), 4),
        }
        try:
            per_class = [round(float(v), 4) for v in r.box.maps.tolist()]
            entry["per_class_map5095"] = dict(zip(CLASS_NAMES, per_class))
        except Exception: pass
        FORMAT_VAL[label] = entry
        print(f"  -> {entry}")
    except Exception as e:
        print(f"  [skip] {e}")

# Doc them per-class cho teacher
print("\n[val] teacher (per-class)")
try:
    r = YOLO(str(teacher_best)).val(data=str(data_yaml), imgsz=IMG_SIZE,
                                       conf=VAL_CONF, iou=VAL_IOU, split="val",
                                       device=DEVICE, verbose=False)
    FORMAT_VAL["teacher_pt"] = {
        "format": "teacher_pt",
        "size_mb": round(teacher_best.stat().st_size/1048576, 2),
        "map50":   round(float(r.box.map50), 4),
        "map5095": round(float(r.box.map), 4),
        "precision": round(float(r.box.mp), 4),
        "recall":  round(float(r.box.mr), 4),
        "per_class_map5095": dict(zip(CLASS_NAMES, [round(float(v), 4) for v in r.box.maps.tolist()])),
    }
    print(f"  -> {FORMAT_VAL['teacher_pt']}")
except Exception as e:
    print(f"  [skip teacher] {e}")

(Path(RESULTS_DIR)/"format_validation.json").write_text(json.dumps(FORMAT_VAL, indent=2, ensure_ascii=False))

# Markdown bang chinh
print("\n=== Bang P1.1 (mAP sau export, cung tap val) ===")
print("| Format | Size MB | mAP50 | mAP50-95 | P | R |")
print("|---|---:|---:|---:|---:|---:|")
for k, v in FORMAT_VAL.items():
    print(f"| {k} | {v['size_mb']} | {v['map50']:.4f} | {v['map5095']:.4f} | {v['precision']:.3f} | {v['recall']:.3f} |")

# Per-class bang
print("\n=== Bang P1.3 (per-class mAP50-95) ===")
header = ["Model"] + CLASS_NAMES
print("| " + " | ".join(header) + " |")
print("|" + "|".join(["---:" if c != "Model" else "---" for c in header]) + "|")
for k, v in FORMAT_VAL.items():
    if "per_class_map5095" not in v: continue
    cells = [k] + [f"{v['per_class_map5095'].get(c, 0):.3f}" for c in CLASS_NAMES]
    print("| " + " | ".join(cells) + " |")

## P1.4 -- 5 anh dinh tinh tot\n\nChon 5 anh val co recall = 1.0 va so doi tuong da dang nhat -> render 3 cot Input | Ground Truth | Prediction.

In [ ]:
# =====================
# P1.4 QUALITATIVE FIGURES (good cases)
# =====================
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
import glob

QUAL_DIR = Path(RESULTS_DIR)/"qualitative"; QUAL_DIR.mkdir(exist_ok=True)

# Dung model KD (seed dau tien hoac serving model)
qual_model = YOLO(str(serving_pt))

img_glob = sorted(glob.glob(str(Path(DATA_ROOT)/"valid/images/*"))) or \
           sorted(glob.glob(str(Path(DATA_ROOT)/"val/images/*")))
label_root = Path(img_glob[0]).parent.parent/"labels" if img_glob else None
assert img_glob, "Khong co anh val"

def load_yolo_labels(label_path, W, H):
    if not Path(label_path).exists(): return []
    boxes = []
    for line in Path(label_path).read_text().splitlines():
        p = line.split()
        if len(p) < 5: continue
        cid, xc, yc, w, h = int(p[0]), *(float(x) for x in p[1:5])
        boxes.append((cid, (xc-w/2)*W, (yc-h/2)*H, (xc+w/2)*W, (yc+h/2)*H))
    return boxes

def iou(a, b):
    ix1=max(a[0],b[0]); iy1=max(a[1],b[1]); ix2=min(a[2],b[2]); iy2=min(a[3],b[3])
    iw=max(0,ix2-ix1); ih=max(0,iy2-iy1); inter=iw*ih
    ua=(a[2]-a[0])*(a[3]-a[1])+(b[2]-b[0])*(b[3]-b[1])-inter
    return inter/ua if ua>0 else 0

def draw_boxes(img, boxes, color):
    img = img.copy(); d = ImageDraw.Draw(img)
    for cid, x1, y1, x2, y2 in boxes:
        d.rectangle([x1,y1,x2,y2], outline=color, width=3)
        d.text((x1, max(0,y1-12)), CLASS_NAMES[cid] if cid < len(CLASS_NAMES) else str(cid), fill=color)
    return img

# Quet 200 anh dau, lay good cases (recall=1.0, >=3 objects, da dang class)
N_SAMPLE = min(200, len(img_glob))
scored = []
for ip in img_glob[:N_SAMPLE]:
    lp = label_root/(Path(ip).stem + ".txt")
    img = Image.open(ip); W, H = img.size
    gt = load_yolo_labels(lp, W, H)
    if len(gt) < 2: continue
    res = qual_model.predict(ip, imgsz=IMG_SIZE, conf=0.25, iou=0.45, device=DEVICE, verbose=False)[0]
    preds = []
    if res.boxes is not None and len(res.boxes) > 0:
        for b in res.boxes:
            preds.append((int(b.cls.item()), *b.xyxy[0].tolist()))
    matched = 0
    for g in gt:
        best = 0
        for p in preds:
            if p[0] != g[0]: continue
            best = max(best, iou(p[1:], g[1:]))
        if best >= 0.5: matched += 1
    rc = matched/len(gt); fp = len(preds) - matched
    n_classes = len(set(g[0] for g in gt))
    if rc == 1.0 and fp <= 1:
        scored.append({"img": ip, "gt": gt, "preds": preds,
                        "recall": rc, "fp": fp, "n_gt": len(gt), "n_classes": n_classes})

# Uu tien anh nhieu doi tuong va nhieu class
scored.sort(key=lambda s: (-s["n_classes"], -s["n_gt"]))
top = scored[:N_QUALITATIVE]
print(f"Tim duoc {len(scored)} good cases, lay top {len(top)}")

meta = []
for i, s in enumerate(top, 1):
    img = Image.open(s["img"]).convert("RGB")
    img_gt = draw_boxes(img, s["gt"], "lime")
    img_pr = draw_boxes(img, s["preds"], "red")
    fig, ax = plt.subplots(1, 3, figsize=(15, 5))
    ax[0].imshow(img);   ax[0].set_title("Input"); ax[0].axis("off")
    ax[1].imshow(img_gt); ax[1].set_title(f"GT ({s['n_gt']} objects, {s['n_classes']} classes)"); ax[1].axis("off")
    ax[2].imshow(img_pr); ax[2].set_title(f"Pred (Recall={s['recall']:.2f}, FP={s['fp']})"); ax[2].axis("off")
    out = QUAL_DIR/f"qualitative_{i}.png"
    fig.tight_layout(); fig.savefig(out, dpi=120, bbox_inches="tight"); plt.close(fig)
    meta.append({"index": i, "image": Path(s["img"]).name, "n_gt": s["n_gt"],
                 "n_classes": s["n_classes"], "recall": s["recall"], "fp": s["fp"],
                 "render": out.name})
    print(f"  {i}. {Path(s['img']).name} (n_gt={s['n_gt']}, classes={s['n_classes']})")

(Path(RESULTS_DIR)/"qualitative.json").write_text(json.dumps(meta, indent=2, ensure_ascii=False))

## Bundle ket qua + tai ve

In [ ]:
# =====================
# BUNDLE + DOWNLOAD
# =====================
import datetime, csv
manifest = {
    "generated_at": datetime.datetime.now().isoformat(timespec="seconds"),
    "device": DEVICE_INFO,
    "config": {
        "seeds": SEEDS, "multiseed_epochs": MULTISEED_EPOCHS,
        "val_conf": VAL_CONF, "val_iou": VAL_IOU,
        "n_qualitative": N_QUALITATIVE, "subset_size": SUBSET_SIZE,
    },
    "seeds_completed": {"baseline": list(SEED_RESULTS["baseline"].keys()),
                         "kd": list(SEED_RESULTS["kd"].keys())},
    "formats_validated": list(FORMAT_VAL.keys()),
    "qualitative_cases": len(meta),
}
(Path(RESULTS_DIR)/"manifest.json").write_text(json.dumps(manifest, indent=2, ensure_ascii=False))

# CSV multiseed
with open(Path(RESULTS_DIR)/"seed_results.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["group","seed","precision","recall","map50","map5095","train_time_s"])
    for grp in ["baseline","kd"]:
        for s, r in SEED_RESULTS[grp].items():
            w.writerow([grp, s, r.get("precision"), r.get("recall"),
                        r.get("map50"), r.get("map5095"), r.get("train_time_s")])

# CSV format
with open(Path(RESULTS_DIR)/"format_validation.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["format","size_mb","map50","map5095","precision","recall"] + [f"class_{c}" for c in CLASS_NAMES])
    for k, v in FORMAT_VAL.items():
        pc = v.get("per_class_map5095", {})
        w.writerow([k, v.get("size_mb"), v.get("map50"), v.get("map5095"),
                    v.get("precision"), v.get("recall")] +
                   [pc.get(c, "") for c in CLASS_NAMES])

# Copy checkpoint mau tu seed dau tien
keep_dir = Path(RESULTS_DIR)/"seed_checkpoints"; keep_dir.mkdir(exist_ok=True)
for grp in ["baseline","kd"]:
    for s, r in SEED_RESULTS[grp].items():
        ck = Path(r["checkpoint"])
        if ck.exists(): shutil.copy2(ck, keep_dir/f"{grp}_s{s}_best.pt")

# Copy ONNX + Engine
for f in [Path(ARTIFACT_DIR)/"serving_model.onnx", Path(ARTIFACT_DIR)/"serving_model.engine"]:
    if f.exists(): shutil.copy2(f, Path(RESULTS_DIR)/f.name)

# Zip + download
zip_path = "/content/p1_results.zip"
if Path(zip_path).exists(): Path(zip_path).unlink()
run(f"cd /content && zip -r {zip_path} p1_results", check=False)
try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print("Tai thu cong tu /content/p1_results.zip:", e)

print("\n=== Hoan tat ===")
print("p1_results.zip chua:")
print("  - seed_results.{json,csv} + seed_checkpoints/*.pt (6 checkpoint)")
print("  - format_validation.{json,csv} (PT, ONNX, TensorRT + per-class)")
print("  - qualitative/*.png + qualitative.json (5 anh dinh tinh)")
print("  - serving_model.onnx, serving_model.engine")
print("  - manifest.json")

## Sau khi tai ve `p1_results.zip`

Giai nen vao `hpc_nhom1_code/reports/p1/` va dung trong bao cao:

1. **P1.1 -- Bang validate sau export** (chen vao Muc 4.4 hoac 4.5):
   `format_validation.csv` -> bang LaTeX co `pytorch_pt / onnx / trt_fp16` voi mAP50, mAP50-95, P, R, Size.

2. **P1.2 -- Bang multi-seed mean +- std** (thay Bang `tab:quality` o Muc 4.4 hoac bo sung bang phu):
   `seed_results.csv` -> bang `Student baseline / Student KD` voi mean +- std qua 3 seed; cot "Delta" hien paired difference KD - baseline.

3. **P1.3 -- Per-class mAP** (chen vao Muc 4.4):
   Lay 5 cot `class_*` cua `format_validation.csv` -> 1 bang nho 5 cot cho moi mo hinh.

4. **P1.4 -- Anh dinh tinh** (chen vao Muc 3.8 hoac 4.4):
   5 file `qualitative_*.png` lam mot figure 5 hang side-by-side.

Sau khi co cac so, tat ca cum tu "can do them seed", "chua validate ONNX/TensorRT" trong threat-to-validity Muc 4.7.4 deu co the chuyen thanh tuyen bo da xac thuc.
